In [2]:
import cv2
import numpy as np
import mediapipe as mp

# Initialize Mediapipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    max_num_hands=2
)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

canvas = None

# Define colors
RIGHT_HAND_COLOR = (255, 0, 0)   # Blue
LEFT_HAND_COLOR  = (0, 0, 255)   # Red

prev_points = {
    "Right": None,
    "Left": None
}

def fingers_up(hand_landmarks):
    tips = [8, 12, 16, 20]
    fingers = []

    for tip in tips:
        if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[tip - 2].y:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers

while True:
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    h, w, c = frame.shape

    if canvas is None:
        canvas = np.zeros_like(frame)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks and results.multi_handedness:
        for handLms, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):

            label = handedness.classification[0].label  # "Left" or "Right"

            finger_state = fingers_up(handLms)

            # Get index fingertip
            x = int(handLms.landmark[8].x * w)
            y = int(handLms.landmark[8].y * h)

            # If open palm → erase
            if sum(finger_state) == 4:
                canvas = np.zeros_like(frame)
                prev_points["Right"] = None
                prev_points["Left"] = None

            # Draw only when index finger is up alone
            elif finger_state == [1,0,0,0]:

                if label == "Right":
                    color = RIGHT_HAND_COLOR
                else:
                    color = LEFT_HAND_COLOR

                if prev_points[label] is None:
                    prev_points[label] = (x, y)

                cv2.line(canvas, prev_points[label], (x, y), color, 6)
                prev_points[label] = (x, y)

            else:
                prev_points[label] = None

            mp_draw.draw_landmarks(frame, handLms, mp_hands.HAND_CONNECTIONS)

    # Merge drawing with frame
    gray_canvas = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    _, inv = cv2.threshold(gray_canvas, 20, 255, cv2.THRESH_BINARY_INV)
    inv = cv2.cvtColor(inv, cv2.COLOR_GRAY2BGR)
    frame = cv2.bitwise_and(frame, inv)
    frame = cv2.bitwise_or(frame, canvas)

    # UI
    cv2.putText(frame, "RIGHT Index = Blue", (10,40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,0,0), 2)
    cv2.putText(frame, "LEFT Index = Red", (10,80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
    cv2.putText(frame, "Open Palm = Erase", (10,120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)

    cv2.imshow("Advanced Virtual Air Canvas", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()